In [ ]:
import torch
import sys
import importlib
import os
from transformers import AutoTokenizer, AutoModel, BertConfig

# Clear Triton cache to force recompilation with fixed code
import shutil
triton_cache = os.path.expanduser("~/.triton/cache")
if os.path.exists(triton_cache):
    shutil.rmtree(triton_cache, ignore_errors=True)
    os.makedirs(triton_cache, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)
# Load config explicitly to avoid config_class mismatch with transformers >= 4.30
config = BertConfig.from_pretrained("zhihan1996/DNABERT-2-117M")
model = AutoModel.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True, config=config)

# Force reload any flash_attn modules that were loaded
flash_attn_modules = [k for k in list(sys.modules.keys()) if 'flash_attn' in k.lower()]
for mod_name in flash_attn_modules:
    if mod_name in sys.modules:
        try:
            importlib.reload(sys.modules[mod_name])
        except:
            pass

model = model.to("cuda")

In [6]:
dna = "ACGTAGCATCGGATCTATCTATCGACACTTGGTTATCGATCTACGAGCATCTCGTTAGC"

inputs = tokenizer(dna, return_tensors = 'pt')["input_ids"].to("cuda")
hidden_states = model(inputs)[0] # [1, sequence_length, 768]

# embedding with mean pooling
embedding_mean = torch.mean(hidden_states[0], dim=0)
print(embedding_mean.shape) # expect to be 768

# embedding with max pooling
embedding_max = torch.max(hidden_states[0], dim=0)[0]
print(embedding_max.shape) # expect to be 768

torch.Size([768])
torch.Size([768])


In [4]:
import torch

# Original tensor shape (3,)
tensor = torch.tensor([1, 2, 3])

# Add dimension at the beginning (dim=0) -> shape (1, 3)
unsqueezed_0 = tensor.unsqueeze(0)

# Add dimension at the end (dim=1) -> shape (3, 1)
unsqueezed_1 = tensor.unsqueeze(1)

unsqueezed_2 = tensor.unsqueeze(1).unsqueeze(2)

print(f"Original shape: {tensor.shape}")         # Output: Original shape: torch.Size([3])
print(f"Shape after unsqueeze(0): {unsqueezed_0.shape}") # Output: Shape after unsqueeze(0): torch.Size([1, 3])
print(f"Shape after unsqueeze(1): \n{unsqueezed_1}") # Output: Shape after unsqueeze(1): torch.Size([3, 1])
print(f"Shape after unsqueeze(1): \n{unsqueezed_2}") # Output: Shape after unsqueeze(1): torch.Size([3, 1])
# print(torch.matmul(unsqueezed_1, unsqueezed_0).repeat_interleave(2, dim=0))
print(torch.matmul(unsqueezed_1, unsqueezed_0).repeat(1, 1, 2))

Original shape: torch.Size([3])
Shape after unsqueeze(0): torch.Size([1, 3])
Shape after unsqueeze(1): 
tensor([[1],
        [2],
        [3]])
Shape after unsqueeze(1): 
tensor([[[1]],

        [[2]],

        [[3]]])
tensor([[[1, 2, 3, 1, 2, 3],
         [2, 4, 6, 2, 4, 6],
         [3, 6, 9, 3, 6, 9]]])


In [5]:
import pandas as pd

combo_train_path = '/home/zcorn/Projects/BioLLMComposition/data/combo_1and2_train.tsv'
combo_train_df = pd.read_csv(combo_train_path, sep='\t')
print(combo_train_df["target_chainseq"].map(lambda x: len(x.split("/")[0])).value_counts().sort_index())
print(combo_train_df["target_chainseq"].map(lambda x: len(x.split("/")[1])).value_counts().sort_index())


target_chainseq
74     2335
75       17
162       1
173       1
174       3
175    7038
181     945
Name: count, dtype: int64
target_chainseq
9     7988
83     416
85    1936
Name: count, dtype: int64


In [6]:
len(combo_train_df["target_chainseq"][0].split("/")[0])


181

In [14]:
from biollmcomposition.config import DATA_DIR, REPO_DIR




/home/zcorn/Projects/proteinDNA_data
